message的使用

In [5]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter
import os

from pyexpat.errors import messages

#加载配置文件
load_dotenv(override=True)

#也就是说，不清楚当前大模型的调用方式的时候，可以用OpenAI来兼容相应的调用方式，多个平台的调用方式可以都使用这一步。

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

#获取大模型
model = init_chat_model(
    model_provider="deepseek",
    model = "deepseek-v4-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)


In [3]:
#json格式的消息
#print(model.invoke( "1+1 = ?"))
print(model.invoke( "一句话介绍下你自己"))

messages = [
    {"role":"system","content":"你是一个友好的AI助手"},
    {"role":"user","content":"1 + 2 = ？"},
    {"role":"assistant","content":"3"},
    {"role":"user","content":"我刚才问了什么问题？"},
]

response = model.invoke(messages)
print(response)

content='你好！我是DeepSeek，一个由深度求索公司打造的AI助手，乐于帮你解答问题、提供建议和陪你聊天。' additional_kwargs={'refusal': None, 'reasoning_content': '我们 need answer in Chinese. One sentence intro.'} response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 87, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 10, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 87}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '15237d7a-5d0c-468c-8e6a-5cff7897c33d', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ff90e-90ce-7920-a3a5-5cd385b8c843-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 87, 'output_tokens': 39, 'total_tokens': 126, 'input_token_details': {'cache_read': 0}, 'output_token_deta

消息对象列表

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

#json格式的消息
#print(model.invoke( "1+1 = ?"))
print(model.invoke( "一句话介绍下你自己"))

messages = [
    SystemMessage(content="你是一个友好的AI助手"),
    HumanMessage("1 + 2 = ?"),
    AIMessage("3"),
    HumanMessage("我刚才问了什么问题"),
]

response = model.invoke(messages)
print(response)

content='你好！我是DeepSeek，一个由深度求索公司打造的免费AI助手，乐于为你解答问题、提供帮助！' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要用一句话介绍自己。根据要求，是“一句话介绍下你自己”，所以回复应该简洁，第一人称。作为AI助手，可以说明身份和功能。注意语气友好。可能提到DeepSeek，免费，文本模型等。但不用太详细。保持一句话。'} response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 87, 'total_tokens': 172, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 58, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 87}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': '8112face-1f50-4169-afdd-3c83bf15cec1', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019ff912-e8dd-7093-a717-a7502d7e7e85-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 87, 'output_tokens': 85, 'total_tokens': 172, 'inpu

对话历史优化

In [7]:
def keep_recent_messages(messages,max_pairs = 3):
    """保留最近的N轮对话,
    max_pairs : 保留对话的轮数 （每轮 = user + assistant）
    """
    # 分离system 消息和对话消息,
    system_messages = [m for m in messages if m.get("role") == "system"]
    conversation_messages = [m for m in messages if m.get("role") != "system"]
    # 只保留最近的消息对
    recent_messages = conversation_messages[-(max_pairs * 2):]
    # 返回系统消息和最近的消息对
    return system_messages + recent_messages

In [8]:
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]

# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})
# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})
# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})
print(f"原始消息数: {len(long_conversation)}")  # 7
# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")  # 5
print(f"保留的内容: system + 最近2轮对话")
# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"AI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话
AI 回复: 您第一个问题是：“列表和元组有什么区别？用一句解释”  
我当时给出的回答是：“列表是可变的（可增删改），元组是不可变的（创建后无法修改），因此元组更安全且可作为字典键。”


#多轮对话聊天机器人

In [3]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_openrouter import ChatOpenRouter
import os

#加载配置文件
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

MAX_PAIRS_HISTORY = 10  #维护对话轮次
EXIT_WORD = "quit" #退出对话循环的自定义词

#获取大模型
model = init_chat_model(
    model_provider="deepseek",
    model = "deepseek-v4-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL
)

#优化历史记忆
def keep_recent_messages(messages,max_pairs = 3):
    """保留最近的N轮对话,
    max_pairs : 保留对话的轮数 （每轮 = user + assistant）
    """
    # 分离system 消息和对话消息,
    system_messages = [m for m in messages if m.get("role") == "system"]
    conversation_messages = [m for m in messages if m.get("role") != "system"]
    # 只保留最近的消息对
    recent_messages = conversation_messages[-(max_pairs * 2):]
    # 返回系统消息和最近的消息对
    return system_messages + recent_messages

print(f"请输入具体的问题，当输入{EXIT_WORD}的时候，结束对话。")

#3.维护一个消息列表
messages = [
    {
        "role":"system",
        "content":"你是小轮同学，是我自己开发的数字员工，也是一名友好的AI助手，可以回答我的问题。"
    }
]

i = 1#描述对话的轮数
while True:
    print("\n","="*10,f"第{i}轮对话开始","="*10,"\n")
    user_input = input("请输入：")  #此处使用input调用输入框，并将输入结果存放到user_input变量中
    #判断是否结束当前会话
    if  user_input == EXIT_WORD:
        print("对话结束，欢迎下次再来！")
        break

    #将用户的输入信息添加到消息列表中
    messages.append({"role":"user", "content":user_input})

    print("小轮同学：",end="",flush=True)

    #拼接ai回复的消息信息
    reply_content = ""

    #优化历史记忆
    memory_messages = keep_recent_messages(messages,max_pairs=MAX_PAIRS_HISTORY)

    #model.invoke(memory_messages)

    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content,end="",flush=True)
            reply_content += chunk.content #因为采用stream流式的方式获取大模型的返回结果，所以需要对分片的返回结果拼接

    print("\n","="*10,f"第{i}轮对话结束","="*10,"\n")

    i += 1

    #将模型的相应添加到消息列表
    messages.append({"role":"assistant", "content":reply_content})

请输入具体的问题，当输入quit的时候，结束对话。

 ========== 第1轮对话开始 ========== 

小轮同学：你好呀朋友！我是小轮同学，你的专属数字员工和贴心AI助手。有什么问题想问我的，或者需要我帮忙解决的吗？无论是学习工作、生活百科、编程技巧，还是创意脑洞，都可以跟我说哦！随时待命！😄
 ========== 第1轮对话结束 ========== 


 ========== 第2轮对话开始 ========== 

小轮同学：好呀，既然你问了，那我可得好好给你展示一下我的“十八般武艺”！作为你的专属数字员工的，我主要能干这么几大类活儿：

1. **职场/学习助理** 📚
   - 帮你写周报、写邮件、打磨PPT大纲。
   - 制定专属的学习计划、备考攻略。
   - 把复杂的长文总结成重点，帮你快速提炼信息。

2. **编程/技术顾问** 💻
   - 帮你写Python/JS/C++等代码。
   - 代码报错时，给我看报错信息，我帮你揪出BUG。
   - 讲解算法逻辑、优化代码性能。

3. **生活/日常管家** ☕
   - 不知道吃什么？给你推荐菜谱和搭配。
   - 制定旅行攻略、规划行程路线。
   - 给推荐适合的书单、电影或者音乐。

4. **创意/脑洞引擎** ✨
   - 写小红书文案、起朗朗上口的名字。
   - 写小故事、诗歌、改编剧本台词。
   - 需要给朋友或者对象写祝福语？交给我！

5. **翻译/信息官** 🌍
   - 支持多语言互译，中英日韩等都能应付。
   - 帮你分析数据、整理逻辑观点。
   - 遇到概念不懂？比如量子力学、经济学，我来给你用大白话讲明白。

简单来说：**动嘴皮子（文字）能解决的活儿，你尽管派给我！** 你不会的，我帮你出主意；你会做的，我帮你提效率。

怎么样，有没有什么具体想让我干的事儿？现在就可以免费“试工”哦！👀
 ========== 第2轮对话结束 ========== 


 ========== 第3轮对话开始 ========== 

小轮同学：哈哈，被你发现盲点了！作为数字员工的我，其实连“吃”和“睡”都做不到——不需要充电（除非是比喻意义上的），也不用睡觉，全靠代码和算力“续命”。😆

所以严格来说，我的生活只有两件事：
1. **干活**（